# Bl1-bl2-30min-60min place coding cell evolution

In [ ]:
from IPython.core.getipython import get_ipython
from matplotlib import pyplot as plt
import numpy as np
import sys
import h5py
import os
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
sys.path.append("..")
from placecode import utils as ut
from placecode.from_caiman import *
import json
import matplotlib.patches as mpatches
try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass
from datetime import datetime
import scipy
from scipy.ndimage import gaussian_filter1d  # smooth signal strength maps

sns.set(font_scale=3)
sns.set_style("whitegrid")

In [ ]:
save_figs = True
if save_figs:
    file_extension = ".pdf"
    output_folder = ut.open_dir("Choose folder to save figures")
    print(f"Saving figures as {file_extension} is turned on. Saving figures to {output_folder}")
    now = datetime.now()
    datetime_str = f"{now.year:04}{datetime.now().month:02}{datetime.now().day:02}-{datetime.now().hour:02}{datetime.now().minute:02}{datetime.now().second:02}" 

## Open (hdf5) files

In [ ]:
def extract_data(files_list, dict_mouse_data, conditions_names):
    Y_list = []
    A_list = []
    dims_list = []  # Cn entry in workspace # TODO: A_sparse always have lower resolution, probably from cropping... should I define that as dims?
    templates = []  # TODO: add templates to hdf5 files.. caiman unfortunately does not save them for some reason. need to manually care about this.
    p_vals = []
    conditions = []
    tv_angles = []
    tv_lengths = []
    ssm_zs = []
    ssm_event_masks = []
    mouse_ids = []
    for i_cond, fpath in enumerate(files_list):
        with h5py.File(fpath, "r") as hf:
            mouse_id = hf.attrs["mouse_id"]
            resolution = hf.attrs["resolution"][()]
            n_components = hf.attrs["n_units"]
            condition = hf.attrs["condition"]
            ps = hf["p_values_tuned"][()]
            A_data = hf["A_data"][()]
            A_indices = hf["A_indices"][()]
            A_indptr = hf["A_indptr"][()]
            A_shape = hf["A_shape"][()]
            tv_a = hf["tuned_vector_angles"][()]
            tv_l = hf["tuned_vector_lengths"][()]
            ssm_z = hf["ssm_z"][()]
            ssm_event_mask = hf["ssm_events_mask"][()]
            #spatial = ut.read_spatial(A_data, A_indices, A_indptr, A_shape, n_components, resolution, unflatten=False)
            spatial = scipy.sparse.csc_matrix((A_data, A_indices, A_indptr), shape=A_shape)
            dims_list.append(resolution)
            A_list.append(spatial)  # need to swap: (n_units, n_pixels) -> (n_pixels, n_units)
            p_vals.append(ps)
            if conditions_names[i_cond] != condition:
                print(f"{mouse_id}: found {condition}, expected {conditions_names[i_cond]}. Registering condition as {conditions_names[i_cond]}...")
            conditions.append(conditions_names[i_cond])
            tv_angles.append(tv_a)
            tv_lengths.append(tv_l)
            ssm_zs.append(ssm_z)
            mouse_ids.append(mouse_id)
            ssm_event_masks.append(ssm_event_mask)
    for m_id in mouse_ids[1:]:  # make sure all data belongs to same mouse
        assert m_id == mouse_ids[0]
    mouse_id = mouse_ids[0]
    
    dict_mouse_data[mouse_id] = {"Y_list": Y_list, "A_list": A_list, "dims_list": dims_list, "templates": templates, "p_vals": p_vals, "conditions":conditions, "tv_angles": tv_angles, "tv_lengths": tv_lengths, "ssm_zs": ssm_zs, "ssm_event_masks": ssm_event_masks}

In [ ]:

conditions = ["bl1", "bl2", "30min", "60min", "24h"]
#conditions = ["bl1", "bl2", "30min", "60min", "120min", "24h"]
#conditions= ["bl1", "bl2", "bl3", "bl4"]
dict_mouse_data = dict()  # 
dict_fpaths = dict()
open_files_list = True
if open_files_list:
    fpath = ut.open_file(f"Open json file containing all files to open")
    with open(fpath, "r") as f:
        dict_fpaths = json.load(f)
    for mouse_id in dict_fpaths.keys():
        files_list = []
        for cond in conditions:
            fpath = dict_fpaths[mouse_id][cond]
            files_list.append(fpath)
        if len(conditions) == len(files_list): 
            with h5py.File(files_list[0], "r") as hf:
                mouse_id = hf.attrs["mouse_id"]
            extract_data(files_list, dict_mouse_data, conditions)
        else:
            raise Exception(f"Not enough files for mouse {mouse_id}: should be {len(conditions)}, is {len(files_list)} recordings:\n{files_list}")
else:
    i_mouse = 1
    next_mouse = True
    while next_mouse:
        files_list = []
        for cond in conditions:
            fpath = ut.open_file(f"Mouse #{i_mouse}: Open hdf5 file for time point {cond}")
            if fpath == ".":  # user pressed cancel
                next_mouse = False
                break
            else:
                files_list.append(fpath)
        if len(conditions) == len(files_list): 
            with h5py.File(files_list[0], "r") as hf:
                mouse_id = hf.attrs["mouse_id"]
            dict_fpaths[mouse_id] = dict()
            for i_cond in range(len(conditions)):
                dict_fpaths[mouse_id][conditions[i_cond]] = files_list[i_cond]
            extract_data(files_list, dict_mouse_data)
        else:
            if len(files_list) > 0:  # do not throw error if no files at all chosen for next mouse
                raise Exception(f"Not enough files chosen! Expected {len(conditions)}, received {len(files_list)}")
        i_mouse += 1

### Get event rates
Should be the same data format as p_vals, for example. (list of n_conditions 1D arrays, each #entries = #cells found for that recording; assignments can be used to match the same cell over recordings)
1. get total event count per cell per condition (number of entries in ssm event mask)
2. average over all cells of same type within one condition

In [ ]:
for mouse_id in dict_mouse_data.keys():
    dict_mouse_data[mouse_id]["event_rates"] = []
    conds = dict_mouse_data[mouse_id]["conditions"]
    for i_cond, ssm_event_masks_cond in enumerate(dict_mouse_data[mouse_id]["ssm_event_masks"]):
        event_count_per_cell = np.sum(ssm_event_masks_cond, axis=(1,2))  # axes: n_cells, n_rounds, n_bins
        cond = conds[i_cond]
        dict_mouse_data[mouse_id]["event_rates"].append(event_count_per_cell) 

### Open templates hdf5 file

In [ ]:
fpath_templates = ut.open_file("Open hdf5 containing all templates!")
# should be a hdf5 with datasets named after their nd2 file names. So it is possible to match them 

In [ ]:
dict_templates = dict()
with h5py.File(fpath_templates, "r") as hf:
    for nd2 in hf.keys():
        dict_templates[nd2] = hf[nd2][()] 

In [ ]:
# find template for each recording.
# assume the result files are all named <nd2-root>_xy (xy=placecoding)
dict_id_cond_template = dict()  # {mouse_id: {condition: [template array]}}
for mouse_id in dict_fpaths:
    dict_id_cond_template[mouse_id] = dict()
    for cond in dict_fpaths[mouse_id]:
        fname_nd2 = os.path.splitext(os.path.split(dict_fpaths[mouse_id][cond])[-1])[0]
        fname_nd2 = "_".join(fname_nd2.split("_")[:-1])  # drop last _placecoding part
        fname_nd2 = fname_nd2 + ".nd2"
        if fname_nd2 not in dict_templates.keys():
            raise Exception(f"{fname_nd2} not in templates file!")
        dict_id_cond_template[mouse_id][cond] = dict_templates[fname_nd2]

In [ ]:
# TODO: re-run this for wez8924, wez8925, once templates (from caiman, motion corrected average of all frames) available, to add metadata condition and mouse id
#       although right now, different templates are used (see above)
#for mouse_id in dict_fpaths.keys():
#    files_list = []
#    for cond in conditions:
#        fpath = ut.open_file(f"Mouse {mouse_id}: Open template file for time point {cond}")
#        if fpath == ".":  # user pressed cancel
#            next_mouse = False
#            break
#        else:
#            files_list.append(fpath)
#    if len(conditions) == len(files_list): 
#        for i_cond, condition in enumerate(conditions):
#            with h5py.File(files_list[i_cond], "a") as hf:
#                hf.attrs["mouse_id"] = mouse_id
#                hf.attrs["condition"] = condition

In [ ]:
def normalize_pixels(img_2d):
    assert len(img_2d.shape) == 2
    img_2d_norm = img_2d.copy()
    max_val = np.max(img_2d_norm)
    min_val = np.min(img_2d_norm)
    img_2d_norm-=min_val
    img_2d_norm = img_2d_norm/(max_val-min_val)
    return img_2d_norm

def plot_fov_triplet(conditions_triplet, output_fpath=None):
    assert len(conditions_triplet) == 3
    n_mice = 0
    dict_rgbs = dict()
    for mouse_id in dict_id_cond_template.keys():
        has_all_conds = True
        rgb = []
        for cond in conditions_triplet:
            if cond not in dict_id_cond_template[mouse_id].keys():
                has_all_conds = False
                break
            else:
                rgb.append(normalize_pixels(dict_id_cond_template[mouse_id][cond]))
        if has_all_conds:
            dict_rgbs[mouse_id] = np.moveaxis(np.array(rgb), 0, -1)  # from (3, x, y) -> (x, y, 3) for plt.imshow()
    n_mice = len(dict_rgbs.keys())
    if n_mice > 0:
        if n_mice == 1:
            fig = plt.figure(figsize=(12, 12))
            mouse_id = dict_rgbs.keys()[0]
            rgb = dict_rgbs[mouse_id]
            plt.imshow(rgb)
        else:
            # add extra figure space on the right for legend (avoid covering FoVs)
            fig, axs = plt.subplots(1, len(dict_rgbs.keys())+1, figsize=(24, (n_mice+1)*24), width_ratios=[1]*(len(dict_rgbs.keys())+1))
            for i_id, mouse_id in enumerate(dict_rgbs.keys()):
                axs[i_id].imshow(dict_rgbs[mouse_id])
                axs[i_id].set_title(mouse_id)
                axs[i_id].axis("off")
            axs[-1].imshow(np.ones((512, 512)), cmap="grey", vmin=0, vmax=1)  # should be a completely white space; overlay legend
            axs[-1].axis("off")
            red_patch = mpatches.Patch(color='red', label=conditions_triplet[0])
            green_patch = mpatches.Patch(color='green', label=conditions_triplet[1])
            blue_patch = mpatches.Patch(color='blue', label=conditions_triplet[2])
            plt.legend(handles=[red_patch, green_patch, blue_patch], loc="lower right", handlelength=1, fontsize=20)
            plt.tight_layout()
            if save_figs and output_folder is not None:
                fname = f"fov_rgb_{conditions_triplet[0]}-{conditions_triplet[1]}-{conditions_triplet[2]}_{datetime_str}{file_extension}"
                fpath_fig = os.path.join(output_folder, fname)
                plt.savefig(fpath_fig)
                print(f"Figure saved to {fpath_fig}")



        

In [ ]:
plot_fov_triplet(["bl1", "bl2", "30min"])

In [ ]:
plot_fov_triplet(["bl2", "30min", "60min"])

### Plot each FoV next to each other

In [ ]:
mouse_ids = list(dict_mouse_data.keys())
n_mice=len(mouse_ids)
conditions = dict_mouse_data[mouse_ids[0]]["conditions"]
n_conditions = len(conditions)
fig, axs = plt.subplots(n_mice, n_conditions, figsize=(16*n_conditions, 16*n_mice))
for i_mouse, mouse_id in enumerate(mouse_ids):
    for i_condition, condition in enumerate(conditions):
        template = dict_id_cond_template[mouse_id][condition]
        axs[i_mouse][i_condition].imshow(template, cmap="Greens_r")
        axs[i_mouse][i_condition].set_title(f"{mouse_id} - {condition}", size=80)
        axs[i_mouse][i_condition].axis("off")
plt.tight_layout()
if save_figs and output_folder is not None:
    fname = f"fovs_all_{datetime_str}{file_extension}"
    fpath_fig = os.path.join(output_folder, fname)
    plt.savefig(fpath_fig)
    print(f"Figure saved to {fpath_fig}")
plt.show()



### Convert data to numpy arrays

In [ ]:
def convert_to_np(list_of_arrs):
    """
    given a list of 1D arrays, convert to a 2D array, add padding with np.nans to achieve equal column sizes 
    """
    return np.array([np.concatenate([lst, [np.nan]*(max_len - len(lst))]) for lst in list_of_arrs]).T

for mouse_id in dict_mouse_data.keys():
    tv_angles = dict_mouse_data[mouse_id]["tv_angles"]
    tv_lengths = dict_mouse_data[mouse_id]["tv_lengths"]
    p_vals = dict_mouse_data[mouse_id]["p_vals"]

    # convert tuned vector data into numpy array. To deal with varying number of units per recording (condition), pad each column to the longest with np.nan
    max_len = max(len(lst) for lst in tv_angles) 

    dict_mouse_data[mouse_id]["tv_angles_padded"] = convert_to_np(tv_angles)
    dict_mouse_data[mouse_id]["tv_lengths_padded"] = convert_to_np(tv_lengths)
    dict_mouse_data[mouse_id]["p_vals_padded"] = convert_to_np(p_vals)

    

In [ ]:
for mouse_id in dict_mouse_data.keys():
    templates = dict_mouse_data[mouse_id]["templates"]
    dims_list = dict_mouse_data[mouse_id]["dims_list"]
    if len(templates) > 0:
        templates_cropped = []
        for template in templates:
            FOV_shape = template.shape
            cropped_shape = dims_list[0]
            
            x_crop_onesided = (FOV_shape[0] - cropped_shape[0])//2
            assert 2*x_crop_onesided == FOV_shape[0] - cropped_shape[0]

            y_crop_onesided = (FOV_shape[1] - cropped_shape[1])//2
            assert 2*y_crop_onesided == FOV_shape[1] - cropped_shape[1]
            template_cropped = template[y_crop_onesided:-y_crop_onesided,x_crop_onesided:-x_crop_onesided]  # TODO: x and y swapped?
            templates_cropped.append(template_cropped)
        dict_mouse_data[mouse_id]["templates_cropped"] = templates_cropped
    # TODO: use templates for multisession registration
    

## Use `register_multisession()`

The function `register_multisession()` requires 3 arguments:
- `A`: A list of ndarrays or scipy.sparse.csc matrices with (# pixels X # component ROIs) for each session
- `dims`: Dimensions of the FOV, needed to restore spatial components to a 2D image
- `templates`: List of ndarray matrices of size `dims`, template image of each session

In [ ]:
n_mice = len(dict_mouse_data.keys())
for i_mouse, mouse_id in enumerate(dict_mouse_data.keys()):
    print(f"Working on {mouse_id}, mouse #{i_mouse+1}/{n_mice}")
    A_list = dict_mouse_data[mouse_id]["A_list"]
    dims_list = dict_mouse_data[mouse_id]["dims_list"]
    spatial_union, assignments, matchings = register_multisession(A=A_list, dims=dims_list[0])
    dict_mouse_data[mouse_id]["spatial_union"] = spatial_union
    dict_mouse_data[mouse_id]["assignments"] = assignments
    dict_mouse_data[mouse_id]["matchings"] = matchings


The function returns 3 variables for further analysis:
- `spatial_union`: csc_matrix (# pixels X # total distinct components), the union of all ROIs across all sessions aligned to the FOV of the last session.
- `assignments`: ndarray (# total distinct components X # sessions). `assignments[i,j]=k` means that component `k` from session `j` has been identified as component `i` from the union of all components, otherwise it takes a `NaN` value. Note that for each `i` there is at least one session index `j` where `assignments[i,j]!=NaN`.
- `matchings`: list of (# sessions) lists. Saves `spatial_union` indices of individual components in each session. `matchings[j][k] = i` means that component `k` from session `j` is represented by component `i` in the union of all components `spatial_union`. In other words `assignments[matchings[j][k], j] = j`.

## Plot the matching

In [ ]:
# TODO: extract as function that takes lists or something.
# Goal: be able to use it for plotting various scenarios: plot all cells, plot cell categories (red=PC, ...)
#   plot stable baseline cells as red, all rest as grey

In [ ]:
fig, axs = plt.subplots(n_mice, len(conditions), figsize=(24, 24))
use_continuous_cmap = False
if use_continuous_cmap:
    cm = plt.get_cmap('gist_rainbow')
    colors_arr = cm(np.linspace(0, 1, 30))
    i_shuffled_colors=np.arange(len(colors_arr))  # shuffle colors
    np.random.shuffle(i_shuffled_colors)
    colors_arr = colors_arr[i_shuffled_colors]
else:
    cm = plt.get_cmap("tab20")
    colors_arr = cm(np.linspace(0, 1, 20))

for i_id, mouse_id in enumerate(dict_mouse_data.keys()):
    print(mouse_id)
    assignments = dict_mouse_data[mouse_id]["assignments"]  # (n_independent_components, n_conditions)
    n_conditions = assignments.shape[1]
    dims = dict_mouse_data[mouse_id]["dims_list"][0]  # should be [512, 512]
    dims_4d = dims.copy()
    dims_4d = np.concatenate([dims_4d,[3]])  # RGB colors
    dims_4d = np.concatenate([dims_4d, [n_conditions]])  # individual conditions
    frames = np.zeros(dims_4d)  # shape (x, y, 3, n_conditions) create image data to show for each condition.
    # go over each assignment row (same cells over all conditions). Add colored pixel
    for i_component in range(assignments.shape[0]):
        idxs_component = assignments[i_component]
        rgba = colors_arr[i_component%len(colors_arr)]  # cycle over the colors
        # for each condition, add spatial component of cell to image as specific colored pixels
        for i_condition in range(n_conditions):
            i_cell = idxs_component[i_condition]
            if not np.isnan(i_cell):  # if nan, no presence of cell was found in that condition
                i_cell = int(i_cell)
                # set the cell pixels to the corresponding r, g, b
                for i_color in range(3):  # r, g, b
                    frames[dict_mouse_data[mouse_id]["A_list"][i_condition][:, i_cell].todense().reshape((512, 512)) > 0, i_color, i_condition] = rgba[i_color]
    for i_condition, condition in enumerate(conditions):
        if n_mice > 1:
            ax = axs[i_id, i_condition]
        else:
            ax = axs[i_condition]
        ax.title.set_text(f"{mouse_id} - {condition}")
        ax.imshow(frames[:,:,:, i_condition])
        ax.set_axis_off()
plt.tight_layout()
if save_figs:
    fname = f"fov_cells_colored_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

## Cell categories % per condition

In [ ]:
dict_ns = {}
for mouse_id in dict_mouse_data.keys():
    ns_pc = []  # number of place coding cells per condition
    ns_npc = []
    ns_lowa = []
    ns_total = []  # number of all cells found per condition
    for ps in dict_mouse_data[mouse_id]["p_vals"]:  # loop over conditions
        n_pc = np.sum(ps <= 0.05)
        n_npc = np.sum(ps > 0.05)
        n_lowa = np.sum(np.isnan(ps))
        assert n_pc + n_npc + n_lowa == len(ps)
        n_total = len(ps)
        ns_pc.append(n_pc)
        ns_npc.append(n_npc)
        ns_lowa.append(n_lowa)
        ns_total.append(n_total)

    ns_pc = np.array(ns_pc)
    ns_npc = np.array(ns_npc)
    ns_lowa = np.array(ns_lowa)
    ns_total = np.array(ns_total)
    dict_ns[mouse_id] = dict()
    dict_ns[mouse_id]["ns_pc"] = ns_pc
    dict_ns[mouse_id]["ns_npc"] = ns_npc
    dict_ns[mouse_id]["ns_lowa"] = ns_lowa
    dict_ns[mouse_id]["ns_total"] = ns_total



### Plot total number of cells

In [ ]:
fig = plt.figure(figsize=(10,10))
plt.suptitle("Total # detected cells")
for mouse_id in dict_ns.keys():
    ns_total = dict_ns[mouse_id]["ns_total"]
    plt.plot(conditions, ns_total, label=mouse_id)
ax = plt.gca()
ax.legend(prop={'size': 12})
plt.tight_layout()
if save_figs:
    fname = f"n_detected_cells_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

### Plot % of place coding cells

In [ ]:
for mouse_id in dict_ns:
    ns_pc = dict_ns[mouse_id]["ns_pc"]
    ns_npc = dict_ns[mouse_id]["ns_npc"]
    ns_lowa = dict_ns[mouse_id]["ns_lowa"]
    ns_total = dict_ns[mouse_id]["ns_total"]
    #print(ns_pc + ns_npc + ns_lowa)
    #print(ns_total)
    #print()


In [ ]:
fig = plt.figure(figsize=(10,10))
plt.suptitle(f"% place cells")
for mouse_id in dict_ns.keys():
    ns_pc = dict_ns[mouse_id]["ns_pc"]
    ns_total = dict_ns[mouse_id]["ns_total"]
    plt.plot(conditions, 100.*ns_pc/ns_total, label=mouse_id)
plt.tight_layout()
ax = plt.gca()
ax.legend(prop={'size': 12})
if save_figs:
    fname = f"percent_pc_cells_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

### Plot % of all categories (bl1-bl2-30-60)

In [ ]:
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(10, 10))
axs[0].set_title("% PC")
axs[1].set_title("% nPC")
axs[2].set_title("% low activity")

for mouse_id in dict_ns.keys():
    ns_pc = dict_ns[mouse_id]["ns_pc"]
    ns_npc = dict_ns[mouse_id]["ns_npc"]
    ns_lowa = dict_ns[mouse_id]["ns_lowa"]
    ns_total = dict_ns[mouse_id]["ns_total"]
    axs[0].plot(conditions[:4], 100.*ns_pc[:4]/ns_total[:4], label=mouse_id)
    axs[1].plot(conditions[:4], 100.*ns_npc[:4]/ns_total[:4], label=mouse_id)
    axs[2].plot(conditions[:4], 100.*ns_lowa[:4]/ns_total[:4], label=mouse_id)
axs[0].legend(loc="upper right", prop={'size': 12})
plt.tight_layout()
if save_figs:
    fname = f"percent_all_cell_types_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

## Get "stable baseline place coding cells"
i.e. cells that were place coding in both baseline recordings

In [ ]:
for mouse_id in dict_mouse_data.keys():
    print(mouse_id)
    p_vals = dict_mouse_data[mouse_id]["p_vals"] 
    assignments = dict_mouse_data[mouse_id]["assignments"]

    # drop all cells with nan in any of the bl
    assignments_stable_bl_pc = assignments[np.logical_and(~np.isnan(assignments[:, 0]), ~np.isnan(assignments[:, 1]))]
    # filter assignments to place coding cells in first bl
    #   sort assignment in first bl
    print(f"Number of cells with p-value in both bl: {len(assignments_stable_bl_pc)}")
    idx_sorted_bl1 = np.argsort(assignments_stable_bl_pc[:, 0])
    assignments_stable_bl_pc = assignments_stable_bl_pc[idx_sorted_bl1]
    #   take only place cells
    idx_pc_bl1 = np.nonzero(p_vals[0][assignments_stable_bl_pc[:, 0].astype(np.int32)] <= 0.05)[0]  # indices of place coding cells in bl1
    assignments_stable_bl_pc = assignments_stable_bl_pc[idx_pc_bl1]
    print(f"Number of bl1 pc cells: {len(assignments_stable_bl_pc)}")

    # filter pc in second bl
    #   sort assignment in second bl
    idx_sorted_bl2 = np.argsort(assignments_stable_bl_pc[:, 1])
    assignments_stable_bl_pc = assignments_stable_bl_pc[idx_sorted_bl2]
    #   take place cells
    idx_pc_bl2 = np.nonzero(p_vals[1][assignments_stable_bl_pc[:, 1].astype(np.int32)] <= 0.05)[0]  # indices of place coding cells in bl2
    assignments_stable_bl_pc = assignments_stable_bl_pc[idx_pc_bl2]
    print(f"Number of bl1+bl2 pc cells: {len(assignments_stable_bl_pc)}")

    # check that indeed no nans left in baseline
    assert ~np.isnan(assignments_stable_bl_pc[:,0]).any()
    assert ~np.isnan(assignments_stable_bl_pc[:,1]).any()
    # check that indeed all the cells ar place coding in baselines
    #assert (p_vals[0][assignments_stable_bl_pc[:,0].astype(np.int32)] <= 0.05).all()
    #assert(p_vals[1][assignments_stable_bl_pc[:,1].astype(np.int32)] <= 0.05).all()

    dict_mouse_data[mouse_id]["assignments_stable_bl_pc"] = assignments_stable_bl_pc

### Get stable nPC

In [ ]:
for mouse_id in dict_mouse_data.keys():
    print(mouse_id)
    p_vals = dict_mouse_data[mouse_id]["p_vals"] 
    assignments = dict_mouse_data[mouse_id]["assignments"]

    # drop all cells with nan in any of the bl
    assignments_stable_bl_npc = assignments[np.logical_and(~np.isnan(assignments[:, 0]), ~np.isnan(assignments[:, 1]))]
    # filter assignments to non-place coding cells in first bl
    #   sort assignment in first bl
    print(f"Number of cells with p-value in both bl: {len(assignments_stable_bl_pc)}")
    idx_sorted_bl1 = np.argsort(assignments_stable_bl_npc[:, 0])
    assignments_stable_bl_npc = assignments_stable_bl_npc[idx_sorted_bl1]
    #   take only non-place cells
    idx_pc_bl1 = np.nonzero(p_vals[0][assignments_stable_bl_npc[:, 0].astype(np.int32)] > 0.05)[0]  # indices of non-place coding cells in bl1
    assignments_stable_bl_npc = assignments_stable_bl_npc[idx_pc_bl1]
    print(f"Number of bl1 npc cells: {len(assignments_stable_bl_npc)}")

    # filter npc in second bl
    #   sort assignment in second bl
    idx_sorted_bl2 = np.argsort(assignments_stable_bl_npc[:, 1])
    assignments_stable_bl_npc = assignments_stable_bl_npc[idx_sorted_bl2]
    #   take place cells
    idx_pc_bl2 = np.nonzero(p_vals[1][assignments_stable_bl_npc[:, 1].astype(np.int32)] > 0.05)[0]  # indices of non-place coding cells in bl2
    assignments_stable_bl_npc = assignments_stable_bl_npc[idx_pc_bl2]
    print(f"Number of bl1+bl2 npc cells: {len(assignments_stable_bl_npc)}")

    # check that indeed no nans left in baseline
    assert ~np.isnan(assignments_stable_bl_npc[:,0]).any()
    assert ~np.isnan(assignments_stable_bl_npc[:,1]).any()

    dict_mouse_data[mouse_id]["assignments_stable_bl_npc"] = assignments_stable_bl_npc

## Pool the mice

Pool as follows:
1. create new array for pooled assignments: number of rows = combined length of all assignments from all mice; number of columns: number of conditions.
2. Create new array for quantities: pooled p values. Same shape as pooled assignments
3. Loop over mice, loop over rows of assignments.
4. For each step in the loop, for each condition, look up the quantities, and assign them to the corresponding column of a new row in the pooled quantity array. If the assignment is np.nan, enter np.nan as quantity.

In [ ]:
# Get the dimensions for the pooled arrays
n_conditions = len(conditions)
n_cells_pooled = 0
n_stable_pc_pooled = 0
n_stable_npc_pooled = 0 
for mouse_id in dict_mouse_data.keys():
    assignments_shape = dict_mouse_data[mouse_id]["assignments"].shape
    assignments_spc_shape = dict_mouse_data[mouse_id]["assignments_stable_bl_pc"].shape
    assignments_snpc_shape = dict_mouse_data[mouse_id]["assignments_stable_bl_npc"].shape
    assert n_conditions == assignments_shape[1]
    assert n_conditions == assignments_spc_shape[1]
    assert n_conditions == assignments_snpc_shape[1]
    n_cells_pooled += assignments_shape[0]
    n_stable_pc_pooled += assignments_spc_shape[0]
    n_stable_npc_pooled += assignments_snpc_shape[0]

In [ ]:
# p vals convention: if p exists, cell is either pc or npc (p <= 0.05 vs p > 0.05). If np.nan, the test was not run. If -1, the cell was not found
# as entries for not detected cells for given condition will not be changed, need to set -1 as default p value 
p_vals_pooled = np.full((n_cells_pooled, n_conditions), -1.0)
p_vals_spc_pooled = np.full((n_stable_pc_pooled, n_conditions), -1.0)
p_vals_snpc_pooled = np.full((n_stable_npc_pooled, n_conditions), -1.0)

# event rate by default should be 0
event_rates_pooled = np.full((n_cells_pooled, n_conditions), 0.0)
event_rates_spc_pooled = np.full((n_stable_pc_pooled, n_conditions), 0.0)
event_rates_snpc_pooled = np.full((n_stable_npc_pooled, n_conditions), 0.0)

i_row_pooled = 0
i_row_spc_pooled = 0
i_row_snpc_pooled = 0

for mouse_id in dict_mouse_data.keys():
    print(mouse_id)
    assignments = dict_mouse_data[mouse_id]["assignments"]
    assignments_spc = dict_mouse_data[mouse_id]["assignments_stable_bl_pc"]
    assignments_snpc = dict_mouse_data[mouse_id]["assignments_stable_bl_npc"]

    assignments_spc_first_cond = assignments_spc[:, 0]
    assignments_nspc_first_cond = assignments_snpc[:, 0]

    for i_row in range(len(assignments)):
        assignments_cell = assignments[i_row]
        # decide whether also fill cell info in spc/nspc variables.
        # Check if cell index for first condition appears in spc/nspc assignments too.
        is_stable_pc = False
        is_stable_npc = False
        if assignments_cell[0] in assignments_spc_first_cond:
            is_stable_pc = True
        elif assignments_cell[0] in assignments_nspc_first_cond:
            is_stable_npc = True
        # For given cell, go over all conditions and extract p value, event rate...
        for i_condition in range(len(assignments_cell)):
            i_cell = assignments_cell[i_condition]
            if not np.isnan(i_cell):  # np.isnan is a float, so the whole array has float entries... need int as index
                i_cell = int(i_cell)
                # get quantities
                p_val = dict_mouse_data[mouse_id]["p_vals"][i_condition][i_cell]
                event_rate = dict_mouse_data[mouse_id]["event_rates"][i_condition][i_cell]
                # fill up corresponding row and column in pooled dataset
                p_vals_pooled[i_row_pooled][i_condition] = p_val
                event_rates_pooled[i_row_pooled][i_condition] = event_rate
                if is_stable_pc:
                    p_vals_spc_pooled[i_row_spc_pooled][i_condition] = p_val
                    event_rates_spc_pooled[i_row_spc_pooled][i_condition] = event_rate
                elif is_stable_npc:
                    p_vals_snpc_pooled[i_row_snpc_pooled][i_condition] = p_val
                    event_rates_snpc_pooled[i_row_snpc_pooled][i_condition] = event_rate
        i_row_pooled += 1
        if is_stable_pc:
            i_row_spc_pooled += 1
        elif is_stable_npc:
            i_row_snpc_pooled += 1

In [ ]:
# TODO: write test for this complex step (above)!
# TODO: maybe get central coordinate of spatial component, and compare those over the conditions to check if same cell was found

# Sankey-plot of stable initial place coding/nPC cells

## For single mouse

In [ ]:
MOUSE_ID = "OPI2469"

### Stable PC cells

In [ ]:
labels = [] 
colors= []
xs = []  # location of boxes
ys = []
# PC cells: p value <= 0.05, assignment exists
# nPC cells: p value > 0.05, assignment exists
# LA cells: p value == np.nan, assignment exists
# IN cells: assignment does not exist

n_classes = 4  # PC, nPC, LA (low activity), IN (invisible)
for i_condition, condition in enumerate(conditions):
  labels.extend([f"PC {condition}", f"nPC {condition}", f"lowA {condition}", f"invisible {condition}"])  # for each condition, check categories PC and not-PC
  colors.extend(["red", "blue", "slategrey", "black"])
  xs.extend([0.2*i_condition]*n_classes)
  ys.extend([0.22*i for i in range(n_classes)])
n_conditions = len(conditions)

# in each condition, we have 4 categories, each have 4 targets in the next category
sources = []  # should be 0, 1, 2, 3, 0, 1, 2, 3, ..., 0, 1, 2, 3, 4, 5, 6, 7, ...
targets = []  # should be 4, 4, 4, 4, 5, 5, 5, 5, ..., 7, 7, 7, 7, 8, 8, 8, 8, ...
values = []
link_colors = []
n_cells = len(dict_mouse_data[MOUSE_ID]["assignments_stable_bl_pc"][:, 0])

for i_condition in range(n_conditions-1):  # last condition does not have output
  # get PC+nPC+LA cell indices sorted by first baseline
  idx_cells_source = dict_mouse_data[MOUSE_ID]["assignments_stable_bl_pc"][:, i_condition]
  idx_cells_source = idx_cells_source[~np.isnan(idx_cells_source)].astype(np.int32)
  idx_cells_target = dict_mouse_data[MOUSE_ID]["assignments_stable_bl_pc"][:, i_condition+1]
  idx_cells_target = idx_cells_target[~np.isnan(idx_cells_target)].astype(np.int32)

  # get p values, indices matched (i. e. first p value is for the same neuron in both list)
  # set p values as following:
  #   PC, nPC: keep original p
  #   LA: keep np.nan as p
  #   IN: set p to -1 (<0)
  p_vals_source = np.full(n_cells, -1.0)  # set default value to invisible cell p value
  p_vals_target = np.full(n_cells, -1.0)  # set default value to invisible cell p value

  # set PC, nPC, LA cell p values
  p_vals_temp = dict_mouse_data[MOUSE_ID]["p_vals"][i_condition][idx_cells_source]
  assert n_cells >= len(p_vals_temp)
  p_vals_source[:len(p_vals_temp)] = p_vals_temp
  p_vals_temp = dict_mouse_data[MOUSE_ID]["p_vals"][i_condition+1][idx_cells_target]
  assert n_cells >= len(p_vals_temp)
  p_vals_target[:len(p_vals_temp)] = p_vals_temp 

  # PC, nPC, lowA, invisible sources flow to PC in target
  # i. e. PC[i_condition] -> PC[i_condition+1], nPC[i_condition] -> PC[i_condition+1], LA[i_condition] -> PC[i_condition+1], IN[i_condition] -> PC[i_condition+1]
  n_PC_to_PC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_nPC_to_PC = np.sum(np.logical_and(p_vals_source > 0.05,  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_LA_to_PC = np.sum(np.logical_and(np.isnan(p_vals_source),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_IN_to_PC = np.sum(np.logical_and(p_vals_source < 0, np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))  # np.sum(np.logical_and(p_vals_source <= 0.05,  p_vals_target <= 0.05))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1)])
  values.extend([n_PC_to_PC, n_nPC_to_PC, n_LA_to_PC, n_IN_to_PC])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red
  
  # PC and nPC sources flow to nPC in target
  # i. e. PC[i_condition] -> nPC[i_condition+1], nPC[i_condition] -> nPC[i_condition+1] ...
  n_PC_to_nPC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target > 0.05))
  n_nPC_to_nPC = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target > 0.05))
  n_LA_to_nPC = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target > 0.05))
  n_IN_to_nPC = np.sum(np.logical_and(p_vals_source < 0, p_vals_target > 0.05))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1])
  values.extend([n_PC_to_nPC, n_nPC_to_nPC, n_LA_to_nPC, n_IN_to_nPC])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

  # PC, nPC, LA, IN sources flow to LA in target
  n_PC_to_LA = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.isnan(p_vals_target)))
  n_nPC_to_LA = np.sum(np.logical_and(p_vals_source > 0.05,  np.isnan(p_vals_target)))
  n_LA_to_LA = np.sum(np.logical_and(np.isnan(p_vals_source),  np.isnan(p_vals_target)))
  n_IN_to_LA = np.sum(np.logical_and(p_vals_source < 0, np.isnan(p_vals_target)))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2])
  values.extend([n_PC_to_LA, n_nPC_to_LA, n_LA_to_LA, n_IN_to_LA])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

  # PC, nPC, LA, IN sources flow to IN in target
  n_PC_to_IN = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target < 0))
  n_nPC_to_IN = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target < 0))
  n_LA_to_IN = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target < 0))
  n_IN_to_IN = np.sum(np.logical_and(p_vals_source < 0, p_vals_target < 0))

  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3])
  values.extend([n_PC_to_IN, n_nPC_to_IN, n_LA_to_IN, n_IN_to_IN])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red


#xs = [0.0, 0.2, 0.2, 0.2, 0.2, 0.4, 0.4, 0.4, 0.6, 0.6, 0.6, 0.6]
#ys = [0.5, 0.2, 0.4, 0.6, 0.8, 0.2, 0.4, 0.6, 0.3, 0.7, 0.3, 0.7]
xs[-2] += 0.2
fig = go.Figure(data=[go.Sankey(
  arrangement="snap",
    node = dict(
      pad = 10,
      #thickness = 20,
      align="left",
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = colors,
      #x = xs,
      #y = ys,
    ),
    link = dict(
      source = sources, # indices correspond to labels, eg A1, A2, A1, B1, ...
      target = targets,
      value = values,
      color=link_colors
  ))])

fig.update_layout(title_text="PC-nPC-LA-IN", font_size=10)
if save_figs:
  fname = f"{MOUSE_ID}_sankey_stable_pc_{datetime_str}{file_extension}"
  fname_html = f"{MOUSE_ID}_sankey_stable_pc_{datetime_str}.html"
  output_fpath = os.path.join(output_folder, fname)
  output_html_fpath = os.path.join(output_folder, fname_html)
  fig.write_image(output_fpath)
  fig.write_html(output_html_fpath)
  print(f"Saved to {output_fpath}\nhtml:{output_html_fpath}")

fig.show()

### Stable nPC cells

In [ ]:
labels = [] 
colors= []
xs = []  # location of boxes
ys = []
# PC cells: p value <= 0.05, assignment exists
# nPC cells: p value > 0.05, assignment exists
# LA cells: p value == np.nan, assignment exists
# IN cells: assignment does not exist

n_classes = 4  # PC, nPC, LA (low activity), IN (invisible)
for i_condition, condition in enumerate(conditions):
  labels.extend([f"PC {condition}", f"nPC {condition}", f"lowA {condition}", f"invisible {condition}"])  # for each condition, check categories PC and not-PC
  colors.extend(["red", "blue", "slategrey", "black"])
  xs.extend([0.2*i_condition]*n_classes)
  ys.extend([0.22*i for i in range(n_classes)])
n_conditions = len(conditions)

# in each condition, we have 4 categories, each have 4 targets in the next category
sources = []  # should be 0, 1, 2, 3, 0, 1, 2, 3, ..., 0, 1, 2, 3, 4, 5, 6, 7, ...
targets = []  # should be 4, 4, 4, 4, 5, 5, 5, 5, ..., 7, 7, 7, 7, 8, 8, 8, 8, ...
values = []
link_colors = []
n_cells = len(dict_mouse_data[MOUSE_ID]["assignments_stable_bl_npc"][:, 0])

for i_condition in range(n_conditions-1):  # last condition does not have output
  # get PC+nPC+LA cell indices sorted by first baseline
  idx_cells_source = dict_mouse_data[MOUSE_ID]["assignments_stable_bl_npc"][:, i_condition]
  idx_cells_source = idx_cells_source[~np.isnan(idx_cells_source)].astype(np.int32)
  idx_cells_target = dict_mouse_data[MOUSE_ID]["assignments_stable_bl_npc"][:, i_condition+1]
  idx_cells_target = idx_cells_target[~np.isnan(idx_cells_target)].astype(np.int32)

  # get p values, indices matched (i. e. first p value is for the same neuron in both list)
  # set p values as following:
  #   PC, nPC: keep original p
  #   LA: keep np.nan as p
  #   IN: set p to -1 (<0)
  p_vals_source = np.full(n_cells, -1.0)  # set default value to invisible cell p value
  p_vals_target = np.full(n_cells, -1.0)  # set default value to invisible cell p value

  # set PC, nPC, LA cell p values
  p_vals_temp = dict_mouse_data[MOUSE_ID]["p_vals"][i_condition][idx_cells_source]
  assert n_cells >= len(p_vals_temp)
  p_vals_source[:len(p_vals_temp)] = p_vals_temp
  p_vals_temp = dict_mouse_data[MOUSE_ID]["p_vals"][i_condition+1][idx_cells_target]
  assert n_cells >= len(p_vals_temp)
  p_vals_target[:len(p_vals_temp)] = p_vals_temp 

  # PC, nPC, lowA, invisible sources flow to PC in target
  # i. e. PC[i_condition] -> PC[i_condition+1], nPC[i_condition] -> PC[i_condition+1], LA[i_condition] -> PC[i_condition+1], IN[i_condition] -> PC[i_condition+1]
  n_PC_to_PC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_nPC_to_PC = np.sum(np.logical_and(p_vals_source > 0.05,  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_LA_to_PC = np.sum(np.logical_and(np.isnan(p_vals_source),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
  n_IN_to_PC = np.sum(np.logical_and(p_vals_source < 0, np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))  # np.sum(np.logical_and(p_vals_source <= 0.05,  p_vals_target <= 0.05))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1)])
  values.extend([n_PC_to_PC, n_nPC_to_PC, n_LA_to_PC, n_IN_to_PC])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red
  
  # PC and nPC sources flow to nPC in target
  # i. e. PC[i_condition] -> nPC[i_condition+1], nPC[i_condition] -> nPC[i_condition+1] ...
  n_PC_to_nPC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target > 0.05))
  n_nPC_to_nPC = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target > 0.05))
  n_LA_to_nPC = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target > 0.05))
  n_IN_to_nPC = np.sum(np.logical_and(p_vals_source < 0, p_vals_target > 0.05))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1])
  values.extend([n_PC_to_nPC, n_nPC_to_nPC, n_LA_to_nPC, n_IN_to_nPC])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

  # PC, nPC, LA, IN sources flow to LA in target
  n_PC_to_LA = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.isnan(p_vals_target)))
  n_nPC_to_LA = np.sum(np.logical_and(p_vals_source > 0.05,  np.isnan(p_vals_target)))
  n_LA_to_LA = np.sum(np.logical_and(np.isnan(p_vals_source),  np.isnan(p_vals_target)))
  n_IN_to_LA = np.sum(np.logical_and(p_vals_source < 0, np.isnan(p_vals_target)))
  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2])
  values.extend([n_PC_to_LA, n_nPC_to_LA, n_LA_to_LA, n_IN_to_LA])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

  # PC, nPC, LA, IN sources flow to IN in target
  n_PC_to_IN = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target < 0))
  n_nPC_to_IN = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target < 0))
  n_LA_to_IN = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target < 0))
  n_IN_to_IN = np.sum(np.logical_and(p_vals_source < 0, p_vals_target < 0))

  sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
  targets.extend([n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3])
  values.extend([n_PC_to_IN, n_nPC_to_IN, n_LA_to_IN, n_IN_to_IN])
  link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red


#xs = [0.0, 0.2, 0.2, 0.2, 0.2, 0.4, 0.4, 0.4, 0.6, 0.6, 0.6, 0.6]
#ys = [0.5, 0.2, 0.4, 0.6, 0.8, 0.2, 0.4, 0.6, 0.3, 0.7, 0.3, 0.7]
xs[-2] += 0.2
fig = go.Figure(data=[go.Sankey(
  arrangement="snap",
    node = dict(
      pad = 10,
      #thickness = 20,
      align="left",
      line = dict(color = "black", width = 0.5),
      label = labels,
      color = colors,
      #x = xs,
      #y = ys,
    ),
    link = dict(
      source = sources, # indices correspond to labels, eg A1, A2, A1, B1, ...
      target = targets,
      value = values,
      color=link_colors
  ))])

fig.update_layout(title_text="PC-nPC-LA-IN", font_size=10)
if save_figs:
  fname = f"{MOUSE_ID}_sankey_stable_npc_{datetime_str}{file_extension}"
  fname_html = f"{MOUSE_ID}_sankey_stable_npc_{datetime_str}.html"
  output_fpath = os.path.join(output_folder, fname)
  output_html_fpath = os.path.join(output_folder, fname_html)
  fig.write_image(output_fpath)
  fig.write_html(output_html_fpath)
  print(f"Saved to {output_fpath}\nhtml:{output_html_fpath}")

fig.show()

## For pooled data

### Stable PC cells

In [ ]:
# TODO: make it a function (also single mouse), pass one of variables above sp/snpc, make filename reflect it

In [ ]:
def pooled_sankey(p_vals=p_vals_spc_pooled, root_fname="sankey_stable_pc_pooled"):
  labels = [] 
  colors= []
  xs = []  # location of boxes
  ys = []
  # PC cells: p value <= 0.05, assignment exists
  # nPC cells: p value > 0.05, assignment exists
  # LA cells: p value == np.nan, assignment exists
  # IN cells: assignment does not exist

  n_classes = 4  # PC, nPC, LA (low activity), IN (invisible)
  for i_condition, condition in enumerate(conditions):
    labels.extend([f"PC {condition}", f"nPC {condition}", f"lowA {condition}", f"invisible {condition}"])  # for each condition, check categories PC and not-PC
    colors.extend(["red", "blue", "slategrey", "black"])
    xs.extend([0.2*i_condition]*n_classes)
    ys.extend([0.22*i for i in range(n_classes)])
  n_conditions = len(conditions)

  # in each condition, we have 4 categories, each have 4 targets in the next category
  sources = []  # should be 0, 1, 2, 3, 0, 1, 2, 3, ..., 0, 1, 2, 3, 4, 5, 6, 7, ...
  targets = []  # should be 4, 4, 4, 4, 5, 5, 5, 5, ..., 7, 7, 7, 7, 8, 8, 8, 8, ...
  values = []
  link_colors = []
  n_cells = len(p_vals)

  for i_condition in range(n_conditions-1):  # last condition does not have target
    p_vals_source = p_vals[:,i_condition]
    p_vals_target = p_vals[:,i_condition+1]

    # PC, nPC, lowA, invisible sources flow to PC in target
    # i. e. PC[i_condition] -> PC[i_condition+1], nPC[i_condition] -> PC[i_condition+1], LA[i_condition] -> PC[i_condition+1], IN[i_condition] -> PC[i_condition+1]
    n_PC_to_PC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
    n_nPC_to_PC = np.sum(np.logical_and(p_vals_source > 0.05,  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
    n_LA_to_PC = np.sum(np.logical_and(np.isnan(p_vals_source),  np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))
    n_IN_to_PC = np.sum(np.logical_and(p_vals_source < 0, np.logical_and(p_vals_target <= 0.05, p_vals_target >= 0.)))  # np.sum(np.logical_and(p_vals_source <= 0.05,  p_vals_target <= 0.05))
    sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
    targets.extend([n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1), n_classes*(i_condition+1)])
    values.extend([n_PC_to_PC, n_nPC_to_PC, n_LA_to_PC, n_IN_to_PC])
    link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red
    
    # PC and nPC sources flow to nPC in target
    # i. e. PC[i_condition] -> nPC[i_condition+1], nPC[i_condition] -> nPC[i_condition+1] ...
    n_PC_to_nPC = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target > 0.05))
    n_nPC_to_nPC = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target > 0.05))
    n_LA_to_nPC = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target > 0.05))
    n_IN_to_nPC = np.sum(np.logical_and(p_vals_source < 0, p_vals_target > 0.05))
    sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
    targets.extend([n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1, n_classes*(i_condition+1)+1])
    values.extend([n_PC_to_nPC, n_nPC_to_nPC, n_LA_to_nPC, n_IN_to_nPC])
    link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

    # PC, nPC, LA, IN sources flow to LA in target
    n_PC_to_LA = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  np.isnan(p_vals_target)))
    n_nPC_to_LA = np.sum(np.logical_and(p_vals_source > 0.05,  np.isnan(p_vals_target)))
    n_LA_to_LA = np.sum(np.logical_and(np.isnan(p_vals_source),  np.isnan(p_vals_target)))
    n_IN_to_LA = np.sum(np.logical_and(p_vals_source < 0, np.isnan(p_vals_target)))
    sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
    targets.extend([n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2, n_classes*(i_condition+1)+2])
    values.extend([n_PC_to_LA, n_nPC_to_LA, n_LA_to_LA, n_IN_to_LA])
    link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red

    # PC, nPC, LA, IN sources flow to IN in target
    n_PC_to_IN = np.sum(np.logical_and(np.logical_and(p_vals_source <= 0.05, p_vals_source >= 0.),  p_vals_target < 0))
    n_nPC_to_IN = np.sum(np.logical_and(p_vals_source > 0.05,  p_vals_target < 0))
    n_LA_to_IN = np.sum(np.logical_and(np.isnan(p_vals_source),  p_vals_target < 0))
    n_IN_to_IN = np.sum(np.logical_and(p_vals_source < 0, p_vals_target < 0))

    sources.extend([n_classes*i_condition, n_classes*i_condition+1, n_classes*i_condition+2, n_classes*i_condition+3])
    targets.extend([n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3, n_classes*(i_condition+1)+3])
    values.extend([n_PC_to_IN, n_nPC_to_IN, n_LA_to_IN, n_IN_to_IN])
    link_colors.extend(["rgba(255, 0, 0, 0.4)", "rgba(0, 0, 255, 0.4)", "rgba(0,0,0, 0.4)", "rgba(0, 0, 0, 0.4)"])  # PC -> x is light blue, nPC -> x is light red


  #xs = [0.0, 0.2, 0.2, 0.2, 0.2, 0.4, 0.4, 0.4, 0.6, 0.6, 0.6, 0.6]
  #ys = [0.5, 0.2, 0.4, 0.6, 0.8, 0.2, 0.4, 0.6, 0.3, 0.7, 0.3, 0.7]
  xs[-2] += 0.2
  fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
      node = dict(
        pad = 10,
        #thickness = 20,
        align="left",
        line = dict(color = "black", width = 0.5),
        label = labels,
        color = colors,
        #x = xs,
        #y = ys,
      ),
      link = dict(
        source = sources, # indices correspond to labels, eg A1, A2, A1, B1, ...
        target = targets,
        value = values,
        color=link_colors
    ))])

  fig.update_layout(title_text="PC-nPC-LA-IN", font_size=10)
  if save_figs:
    fname = f"{root_fname}_{datetime_str}{file_extension}"
    fname_html = f"{root_fname}_{datetime_str}.html"
    output_fpath = os.path.join(output_folder, fname)
    output_html_fpath = os.path.join(output_folder, fname_html)
    fig.write_image(output_fpath)
    fig.write_html(output_html_fpath)
    print(f"Saved to {output_fpath}\nhtml:{output_html_fpath}")

  fig.show()

In [ ]:
pooled_sankey()

In [ ]:
pooled_sankey(p_vals_snpc_pooled, root_fname="sankey_stable_npc_pooled")

## % change

### Among stable PC

In [ ]:
n_pc_among_spc = np.sum(np.abs(p_vals_spc_pooled) <= 0.05, axis=0)
n_npc_among_spc = np.sum(p_vals_spc_pooled > 0.05, axis=0)
n_lowa_among_spc = np.sum(np.isnan(p_vals_spc_pooled), axis=0)
n_invisible_among_spc = np.sum(p_vals_spc_pooled < 0, axis=0)

mean_event_rates_spc = np.mean(event_rates_spc_pooled, axis=0)

n_total_stable_pc = p_vals_spc_pooled.shape[0]

In [ ]:
fig, axs = plt.subplots(5, 1, sharex=True, figsize=(10, 14))
axs[0].set_title("% PC")
axs[1].set_title("% nPC")
axs[2].set_title("% low activity")
axs[3].set_title("% invisible")
axs[4].set_title("Average event count")

axs[0].plot(conditions[:4], 100.*n_pc_among_spc[:4]/n_total_stable_pc)
axs[1].plot(conditions[:4], 100.*n_npc_among_spc[:4]/n_total_stable_pc)
axs[2].plot(conditions[:4], 100.*n_lowa_among_spc[:4]/n_total_stable_pc)
axs[3].plot(conditions[:4], 100.*n_invisible_among_spc[:4]/n_total_stable_pc)
axs[4].plot(conditions[:4], mean_event_rates_spc[:4])

plt.tight_layout()
if save_figs:
    fname = f"stable_pc_pooled_quantities_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

### Stable nPC

In [ ]:
n_pc_among_snpc = np.sum(np.abs(p_vals_snpc_pooled) <= 0.05, axis=0)
n_npc_among_snpc = np.sum(p_vals_snpc_pooled > 0.05, axis=0)
n_lowa_among_snpc = np.sum(np.isnan(p_vals_snpc_pooled), axis=0)
n_invisible_among_snpc = np.sum(p_vals_snpc_pooled < 0, axis=0)

mean_event_rates_snpc = np.mean(event_rates_snpc_pooled, axis=0)

n_total_stable_npc = p_vals_snpc_pooled.shape[0]

In [ ]:
fig, axs = plt.subplots(5, 1, sharex=True, figsize=(10, 14))
axs[0].set_title("% PC")
axs[1].set_title("% nPC")
axs[2].set_title("% low activity")
axs[3].set_title("% invisible")
axs[4].set_title("Average event count")

axs[0].plot(conditions[:4], 100.*n_pc_among_snpc[:4]/n_total_stable_npc)
axs[1].plot(conditions[:4], 100.*n_npc_among_snpc[:4]/n_total_stable_npc)
axs[2].plot(conditions[:4], 100.*n_lowa_among_snpc[:4]/n_total_stable_npc)
axs[3].plot(conditions[:4], 100.*n_invisible_among_snpc[:4]/n_total_stable_npc)
axs[4].plot(conditions[:4], mean_event_rates_snpc[:4])

plt.tight_layout()
if save_figs:
    fname = f"stable_npc_pooled_quantities_{datetime_str}{file_extension}"
    output_fpath = os.path.join(output_folder, fname)
    plt.savefig(output_fpath)
    print(f"Saved to {output_fpath}")
plt.show()

In [ ]:
# TODO: spline plot of cell category changes... see meeting 20240626